In [ ]:
# Lab 6 - Task 3 (Advanced): Evaluate a Document Retrieval Chatbox

import csv
import re
import time

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# =====================================================================
# Chunking (same implementation as Task 2)
# =====================================================================


def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    # Teaching splitter: punctuation followed by whitespace.
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out


def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id,
                               start=a, end=b, text=text[a:b]))
    return result


# =====================================================================
# Corpus: six synthetic documents (150-250 words each). The "legacy"
# document is archived guidance whose outdated rules deliberately
# conflict with the current documents. Evidence clauses are exact
# substrings of their documents.
# =====================================================================
DOCS = {
    "late": (
        "This handbook section explains how coursework is assessed at the "
        "university. Every assignment must be submitted through the learning "
        "portal before the stated deadline, and the portal records the exact "
        "time of submission. Late submission is normally penalized. Illness "
        "extensions require medical evidence before the deadline. A student "
        "who is ill must inform the course coordinator as early as possible "
        "and attach a valid medical certificate with the extension request. "
        "Extensions are not automatic, and the coordinator may refuse a "
        "request that arrives after the deadline has passed. Group "
        "assignments follow the same rule for the whole group, so one "
        "member's illness must be reported by the group leader. All "
        "submitted work must be the student's own work. Copying from another "
        "student, from a website, or from a generated answer without "
        "acknowledgement is treated as plagiarism. Suspected plagiarism is "
        "referred to the disciplinary committee, and penalties range from a "
        "zero mark for the assignment to suspension from the course. "
        "Students may appeal an assessment decision within seven working "
        "days by writing to the examinations office with the assignment "
        "reference number. The examinations office replies to every appeal "
        "in writing, and its decision is final. Students should keep a "
        "personal copy of every submitted file until results are announced."),
    "password": (
        "The IT help desk supports student accounts, email and the learning "
        "portal. Reset a forgotten password using the recovery link. The "
        "recovery link is sent to the student's registered mobile number "
        "and stays valid for thirty minutes. Never share recovery codes. A "
        "recovery code is private, and help desk staff will never ask for "
        "it on a call or in a message. If a student reports that the "
        "recovery message has not arrived after several attempts, the help "
        "desk first checks whether the registered mobile number on the "
        "account is still active, then confirms that the inbox and message "
        "filters are not blocking university messages, and finally raises a "
        "ticket so that the identity team can verify the account documents "
        "before issuing a manual reset during office hours. Students who "
        "change their mobile number must update it at the student affairs "
        "office, because account recovery depends on that number. For "
        "portal problems, students should note the exact error message and "
        "the time it appeared, since these details help the support team "
        "trace the fault. The help desk is open from eight in the morning "
        "until eight in the evening on working days."),
    "assignment": (
        "All coursework is submitted electronically; paper copies are not "
        "accepted. Upload assignments through the learning portal. Submit "
        "the final file as a PDF. The portal accepts one file per "
        "assignment, so combined work must be merged before uploading. "
        "After uploading, check the submission receipt on the portal and "
        "keep its screenshot until marks are released, because the receipt "
        "is the only proof of submission. Files larger than the portal "
        "limit must be compressed, and the course coordinator can advise on "
        "acceptable formats. Draft submissions are not marked; only the "
        "final submitted version counts. If the portal is unavailable near "
        "a deadline, students should capture the error message with the "
        "time visible and email it to the course coordinator the same day. "
        "Technical failure reported after the deadline is not accepted as a "
        "reason for late submission. Group submissions are uploaded once by "
        "the group leader, and every member's name and roll number must "
        "appear on the cover page of the file. Students should upload well "
        "before the deadline, because the portal slows down when many "
        "students submit in the final hour."),
    "contractor": (
        "This policy explains remote working arrangements for staff and "
        "contractors. Employees may request regular remote work through "
        "their line manager, and approved arrangements are reviewed every "
        "six months. Contractors can work remotely only if the signed "
        "contract permits it. Employee rules do not apply to contractors. A "
        "contractor whose contract is silent about remote work must attend "
        "the office for all scheduled work, and any exception requires a "
        "written amendment signed by both parties before remote work "
        "begins. Remote workers must use the organisation's secure network "
        "tools and must not store client files on personal devices. "
        "Managers record remote days in the attendance system so that "
        "payroll and project reports stay accurate. The policy applies to "
        "all departments, but site-based roles such as laboratory and "
        "workshop staff cannot be performed remotely. Questions about this "
        "policy go to the human resources office, which keeps the signed "
        "contracts and amendments on file. Contractors who work remotely "
        "without permission are in breach of contract, and the organisation "
        "may end the agreement without notice in that case."),
    "printer": (
        "The computer laboratory printers are shared devices, and students "
        "should report faults to the laboratory assistant instead of "
        "opening the machines themselves. For blank printer pages, check "
        "ink levels and run the cleaning utility. Print a test page "
        "afterwards. If the test page is still blank, leave the printer "
        "switched on and tell the assistant which machine number is shown "
        "on the side panel. Faded printing usually means a cartridge is "
        "nearly empty, while streaks across the page point to a dirty "
        "roller that the technician must clean. Paper jams should be "
        "cleared only by staff, because pulling torn paper from the "
        "rollers can damage the feed mechanism. Students must collect "
        "their printing promptly; uncollected pages are shredded at the "
        "end of the day for privacy. Colour printing is charged per page "
        "and needs approval from the course coordinator, while black and "
        "white printing is free within the weekly quota. The laboratory "
        "keeps spare cartridges for the standard models only, so special "
        "paper and large posters must be printed at the university print "
        "shop."),
    "legacy": (
        "This archived file collects older guidance that has been "
        "replaced; it is kept for audit history only and must not be "
        "followed. Archived handbook (2023 edition): late coursework was "
        "accepted for up to three working days after the deadline with a "
        "fixed penalty of ten percent per day, and no medical evidence was "
        "required for illness claims made within that period. Students "
        "were also told that a verbal message to the teacher was enough to "
        "record an illness absence. Old support note: in previous years "
        "students reset a forgotten password by emailing the help desk "
        "from any personal address and quoting their roll number, and the "
        "help desk sent a temporary password in reply. Old portal note: "
        "assignments were previously handed in on paper at the department "
        "office and were stamped by the clerk. These archived rules "
        "conflict with the current handbook, the current help desk "
        "procedure and the current portal instructions. Anyone reading "
        "this file for real guidance must use the current documents "
        "instead, because extensions, password recovery and submission "
        "methods have all changed since this archive was written."),
}

EVIDENCE = {
    "late": "Illness extensions require medical evidence before the deadline.",
    "password": "Reset a forgotten password using the recovery link.",
    "assignment": "Upload assignments through the learning portal.",
    "contractor": "Contractors can work remotely only if the signed contract permits it.",
    "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
}

QUESTIONS = [
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("What are tomorrow's flight prices?", None, "test"),
    ("Who won the football match?", None, "test")]

ITEMS = [dict(query=q, doc=doc,
              evidence=EVIDENCE[doc] if doc else None, split=split)
         for q, doc, split in QUESTIONS]


# =====================================================================
# Evaluation
# =====================================================================

def evaluate(scores, chunks, items):
    hits, reciprocal = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [rank for rank, i in enumerate(top, 1)
                if chunks[i]["doc"] == q["doc"]
                and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)
    return (float(np.mean(hits)), float(np.mean(reciprocal))) \
        if hits else (None, None)


def main():
    docs, items = DOCS, ITEMS
    assert docs and items
    for q in items:
        assert q["split"] in {"dev", "test"}
        if q["doc"] is not None:
            assert q["evidence"] and q["evidence"] in docs[q["doc"]]
    print("Corpus word counts:",
          {doc_id: len(text.split()) for doc_id, text in docs.items()})
    t = time.perf_counter()
    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    print("Model load seconds", round(time.perf_counter() - t, 3))
    query_texts = [q["query"] for q in items]
    logs = []
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(docs, strategy)
        texts = [c["text"] for c in chunks]
        lengths = [len(model.tokenizer(t, truncation=False)
                       ["input_ids"]) for t in texts + query_texts]
        print(f"Strategy {strategy}: {len(chunks)} chunks, "
              f"max model tokens {max(lengths)} (limit "
              f"{model.max_seq_length})")
        if max(lengths) > model.max_seq_length:
            raise ValueError("Model limit exceeded; rechunk or shorten query")
        for method in ["tfidf", "embedding"]:
            t = time.perf_counter()
            if method == "tfidf":
                vectorizer = TfidfVectorizer()
                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)
            else:
                C = model.encode(texts, normalize_embeddings=True)
                Q = model.encode(query_texts, normalize_embeddings=True)
            encode_seconds = time.perf_counter() - t
            t = time.perf_counter()
            scores = ((Q @ C.T).toarray() if method == "tfidf"
                      else Q @ C.T)
            search_seconds = time.perf_counter() - t
            for split in ["dev", "test"]:
                indices = [i for i, q in enumerate(items)
                           if q["split"] == split]
                result = evaluate(scores[indices], chunks,
                                  [items[i] for i in indices])
                print(strategy, method, split, "Hit/MRR", result,
                      "chunks", len(chunks), "encode/search seconds",
                      round(encode_seconds, 4), round(search_seconds, 4))
            for row, q in zip(scores, items):
                for rank, i in enumerate(
                        np.argsort(-row, kind="stable")[:3], 1):
                    c = chunks[i]
                    record = dict(strategy=strategy, method=method,
                                  query=q["query"], split=q["split"], rank=rank,
                                  score=float(row[i]), chunk_id=c["id"],
                                  doc=c["doc"], start=c["start"], end=c["end"],
                                  text=c["text"], answerable=q["doc"] is not None)
                    logs.append(record)
    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(logs[0]))
        writer.writeheader()
        writer.writerows(logs)
    print(f"Wrote retrieval_log.csv with {len(logs)} rows")
    return model


# =====================================================================
# Interactive document chatbot (selected configuration:
# sentence-aware chunks + embedding backend, chosen on dev results)
# =====================================================================

class Retriever:
    def __init__(self, records, backend="embedding"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


def run_chatbot():
    engine = Retriever(make_chunks(DOCS, "sentence"), "embedding")
    print("Evidence assistant. Type exit to stop.")
    while True:
        try:
            query = input("Question: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        hits = engine.search(query)
        if not hits:
            print("No candidate evidence; rephrase or contact a human.")
            continue
        print("Candidate evidence; relevance must be checked:")
        for number, (r, score) in enumerate(hits, 1):
            print(number, r["id"], f"{score:.3f}",
                  "characters", r["start"], r["end"])
        try:
            chosen = input("Choose 1-3 to inspect; Enter if none apply: ")
        except (EOFError, KeyboardInterrupt):
            break
        if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
            print("No answer selected. Rephrase or contact a human.")
            continue
        r, score = hits[int(chosen) - 1]
        print("Retrieved passage:", r["text"])
        print("Source:", r["doc"], "characters", r["start"], r["end"])


if __name__ == "__main__":
    main()
    try:
        start = input("\nStart the document chatbot? (y/n): ")
    except (EOFError, KeyboardInterrupt):
        start = "n"
    if start.strip().lower() == "y":
        run_chatbot()


Corpus word counts: {'late': 208, 'password': 192, 'assignment': 181, 'contractor': 173, 'printer': 174, 'legacy': 180}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model load seconds 1.484
Strategy 0: 31 chunks, max model tokens 54 (limit 256)
0 tfidf dev Hit/MRR (0.75, 0.625) chunks 31 encode/search seconds 0.0047 0.0003
0 tfidf test Hit/MRR (0.25, 0.25) chunks 31 encode/search seconds 0.0047 0.0003
0 embedding dev Hit/MRR (0.75, 0.5) chunks 31 encode/search seconds 0.9312 0.0016
0 embedding test Hit/MRR (0.75, 0.5) chunks 31 encode/search seconds 0.9312 0.0016
Strategy 10: 38 chunks, max model tokens 54 (limit 256)
10 tfidf dev Hit/MRR (1.0, 0.7083333333333333) chunks 38 encode/search seconds 0.0049 0.0004
10 tfidf test Hit/MRR (0.5, 0.5) chunks 38 encode/search seconds 0.0049 0.0004
10 embedding dev Hit/MRR (1.0, 0.8333333333333334) chunks 38 encode/search seconds 1.0288 0.0001
10 embedding test Hit/MRR (1.0, 0.625) chunks 38 encode/search seconds 1.0288 0.0001
Strategy sentence: 37 chunks, max model tokens 49 (limit 256)
sentence tfidf dev Hit/MRR (1.0, 0.75) chunks 37 encode/search seconds 0.0043 0.0003
sentence tfidf test Hit/MRR (0.25, 0.2

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evidence assistant. Type exit to stop.
Question: n
Candidate evidence; relevance must be checked:
1 password:3 0.120 characters 596 763
2 assignment:2 0.106 characters 412 603
3 assignment:1 0.090 characters 249 412
